# Translate
Takes decoded word sequences and reconstructs natural English sentences using the Gemini API.
See `docs/translate.md` for theory.

## Imports and API setup

In [1]:
import json
import os
from google import genai
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv('GOOGLE_API_KEY')

if not api_key:
    raise ValueError('GOOGLE_API_KEY not found in .env file')

client = genai.Client(api_key=api_key)

print('Gemini API client ready')

Gemini API client ready


## Load dictionary

In [2]:
with open('../data/dictionary.json', 'r') as f:
    dictionary = json.load(f)

sentences    = dictionary['sentences_detected']
sound_to_word = dictionary['sound_to_word']

print(f'sentences to translate: {len(sentences)}')
print()
for i, s in enumerate(sentences):
    print(f'  {i+1}. {" ".join(s["words"])}')

sentences to translate: 5

  1. warning warning am question
  2. warning what what am question
  3. am am danger warning warning warning
  4. what understand am warning what am question
  5. am danger warning am warning ?


## Build the translation prompt
We give Gemini the word sequence and the vocabulary mapping as context, then ask it to reconstruct a natural English sentence.

In [3]:
def build_prompt(word_sequence):
    vocab_str = '\n'.join([f'  {sound} → {word}' for sound, word in sound_to_word.items()])
    words_str = ' '.join(word_sequence)
    return f"""You are decoding an alien language from the novel Project Hail Mary.

Rocky communicates through pressure waves. Each wave pattern maps to an English concept:
{vocab_str}

The following is a sequence of decoded concepts from Rocky's speech:
{words_str}

Reconstruct the most likely natural English sentence Rocky is trying to communicate.
The decoded concepts may be imperfect due to signal noise — use context to infer meaning.
Reply with only the reconstructed sentence, nothing else."""

# preview prompt for sentence 1
print(build_prompt(sentences[0]['words']))

You are decoding an alien language from the novel Project Hail Mary.

Rocky communicates through pressure waves. Each wave pattern maps to an English concept:
  click_low → I
  click_mid → here
  click_high → warning
  hum_low → what
  hum_mid → am
  hum_high → safe
  chirp_up → question
  chirp_down → danger
  chord_simple → understand
  chord_rich → need
  chord_alien → critical

The following is a sequence of decoded concepts from Rocky's speech:
warning warning am question

Reconstruct the most likely natural English sentence Rocky is trying to communicate.
The decoded concepts may be imperfect due to signal noise — use context to infer meaning.
Reply with only the reconstructed sentence, nothing else.


## Translate each sentence

In [6]:
def translate(word_sequence):
    response = client.models.generate_content(
        model='gemini-3.6-flash',
        contents=build_prompt(word_sequence)
    )
    return response.text.strip()

print("Translating Rocky's sentences...\n")
translations = []

for i, sentence in enumerate(sentences):
    words  = sentence['words']
    result = translate(words)
    translations.append(result)
    print(f'Sentence {i+1}')
    print(f'  Raw words  : {" ".join(words)}')
    print(f'  Translation: {result}')
    print()

Translating Rocky's sentences...

Sentence 1
  Raw words  : warning warning am question
  Translation: Am I in danger?

Sentence 2
  Raw words  : warning what what am question
  Translation: What am I?

Sentence 3
  Raw words  : am am danger warning warning warning
  Translation: Warning: I am in danger!

Sentence 4
  Raw words  : what understand am warning what am question
  Translation: Do you understand what I am warning you about?

Sentence 5
  Raw words  : am danger warning am warning ?
  Translation: Am I in danger?



## Compare to known meanings

In [7]:
known = list(dictionary['english_meaning'].values())

print('Translation vs known meaning:\n')
print(f'{"#":3}  {"Translation":40}  {"Known meaning"}')
print('-' * 75)
for i, (translation, meaning) in enumerate(zip(translations, known)):
    print(f'{i+1:3}  {translation:40}  {meaning}')

Translation vs known meaning:

#    Translation                               Known meaning
---------------------------------------------------------------------------
  1  Am I in danger?                           Hello / I am here
  2  What am I?                                What is this?
  3  Warning: I am in danger!                  Warning! Danger!
  4  Do you understand what I am warning you about?  I understand
  5  Am I in danger?                           I need help


## Save translations to dictionary

In [8]:
for i, sentence in enumerate(dictionary['sentences_detected']):
    sentence['translation'] = translations[i]

with open('../data/dictionary.json', 'w') as f:
    json.dump(dictionary, f, indent=2)

print('saved  data/dictionary.json  with translations')
print()
print('Pipeline complete:')


saved  data/dictionary.json  with translations

Pipeline complete:
